# Module 3 — PubMed Research Trend Analysis

This notebook uses **Biopython's Entrez module** to query NCBI PubMed and count how many papers have been published each year on endophyte bioactivity topics from 2000 to 2024.

This demonstrates programmatic access to biological databases — a core bioinformatics skill.

**This notebook is independent — you can run it without running notebooks 01 or 02 first.**

In [ ]:
# ── CELL 1: Install Biopython ──
!pip install biopython --quiet
print("✓ Biopython installed")

In [ ]:
# ── CELL 2: Setup ──
from Bio import Entrez
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import time
import os

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# NCBI requires an email to use their API — put yours here
Entrez.email = "banerjee.shruti1306@gmail.com"

print("✓ Ready")

In [ ]:
# ── CELL 3: Define query function ──
def get_pubmed_count(search_term, year):
    """
    Query PubMed for number of papers matching
    a search term within a specific publication year.
    """
    query = f"({search_term}) AND {year}[PDAT]"
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=0)
        record = Entrez.read(handle)
        handle.close()
        return int(record["Count"])
    except Exception as e:
        print(f"  [error] {year}: {e}")
        return 0

# Quick test
test = get_pubmed_count("endophyte AND anticancer", 2020)
print(f"✓ Test query works. Papers in 2020 for 'endophyte AND anticancer': {test}")

In [ ]:
# ── CELL 4: Run all queries ──
# This makes ~75 API calls. Takes 2-3 minutes. Do not interrupt.

search_terms = {
    "Anticancer":   "endophyte AND anticancer",
    "Antimicrobial": "endophyte AND antimicrobial",
    "Antifungal":   "endophyte AND antifungal"
}

years   = list(range(2000, 2025))
results = {label: [] for label in search_terms}

for label, term in search_terms.items():
    print(f"\nQuerying: '{term}'")
    for year in years:
        count = get_pubmed_count(term, year)
        results[label].append(count)
        print(f"  {year}: {count}", end="  ")
        time.sleep(0.4)  # respect NCBI rate limit (3 requests/sec max)

print("\n\n✓ All queries complete!")

In [ ]:
# ── CELL 5: Build DataFrame and save ──
df_trends = pd.DataFrame(results, index=years)
df_trends.index.name = "Year"
df_trends.to_csv("data/pubmed_trends.csv")
print("✓ Saved to data/pubmed_trends.csv")
df_trends

In [ ]:
# ── CELL 6: Plot trends ──
fig, ax = plt.subplots(figsize=(13, 6))

palette = {
    "Anticancer":    "#C62828",
    "Antimicrobial": "#1565C0",
    "Antifungal":    "#2E7D32"
}

for label, color in palette.items():
    ax.plot(
        df_trends.index, df_trends[label],
        marker="o", markersize=5,
        linewidth=2.2, color=color, label=label
    )
    ax.fill_between(df_trends.index, df_trends[label], alpha=0.07, color=color)

ax.set_title("PubMed Publications on Endophyte Bioactivity (2000–2024)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Number of publications", fontsize=11)
ax.legend(fontsize=10, title="Search term", title_fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax.grid(True, alpha=0.25, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/pubmed_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved to figures/pubmed_trends.png")

In [ ]:
# ── CELL 7: Growth statistics ──
print("=== Publication Growth: 2000 → 2024 ===")
print()
for label in search_terms:
    start = df_trends.loc[2000, label]
    end   = df_trends.loc[2024, label]
    peak_year = df_trends[label].idxmax()
    peak_val  = df_trends[label].max()
    if start > 0:
        growth = ((end - start) / start) * 100
        print(f"{label}")
        print(f"  2000: {start} papers  →  2024: {end} papers  ({growth:+.0f}% growth)")
        print(f"  Peak: {peak_val} papers in {peak_year}")
    else:
        print(f"{label}: {start} → {end} papers (baseline was 0)")
    print()

In [ ]:
# ── CELL 8: Download outputs ──
from google.colab import files
files.download("data/pubmed_trends.csv")
files.download("figures/pubmed_trends.png")
print("✓ Downloaded! Upload to your GitHub repo under data/ and figures/")